In [0]:
#Criando o schema da camada Silver para organizar os dados tratados
spark.sql("CREATE SCHEMA IF NOT EXISTS workspace.silver")

In [0]:
#Lendo a tabela bruta de informações para começar o tratamento na Silver
df_info = spark.table("workspace.bronze.tb_info_filmes")

display(df_info.limit(10))

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

#Mantendo a versão mais recente de cada filme caso ele tenha sido inserido mais de uma vez. Como a Bronze usa append, escolho a carga mais recente para manter uma linha atual por filme
janela = Window.partitionBy("id").orderBy(F.col("ingestion_datetime").desc())

df_info_silver = (
    df_info
    .withColumn("ordem", F.row_number().over(janela))
    .filter(F.col("ordem") == 1)
    .drop("ordem"))

#Padronizando o texto do status antes de traduzir
df_info_silver = df_info_silver.withColumn(
    "status_normalizado",
    F.lower(F.trim(F.regexp_replace(F.col("status"), r"[-_]+", " "))))

#Traduzindo os status conhecidos e deixando os inválidos como "Não Informado"
df_info_silver = df_info_silver.withColumn(
    "status_filme",
    F.when(F.col("status_normalizado") == "released", "Lançado")
    .when(F.col("status_normalizado") == "post production", "Pós-Produção")
    .when(F.col("status_normalizado") == "in production", "Em Produção")
    .when(F.col("status_normalizado") == "planned", "Planejado")
    .when(F.col("status_normalizado") == "rumored", "Rumores")
    .when(F.col("status_normalizado") == "canceled", "Cancelado")
    .otherwise("Não Informado"))

display(df_info_silver.select("id", "status", "status_filme").limit(10))

In [0]:
#Tentando os formatos de data encontrados no arquivo. Caso nenhum funcionar, a data fica nula em vez de interromper o tratamento
df_info_silver = df_info_silver.withColumn(
    "data_lancamento",
    F.coalesce(
        F.expr("try_to_date(release_date, 'yyyy-MM-dd')"),
        F.expr("try_to_date(release_date, 'MM-dd-yyyy')"),
        F.expr("try_to_date(release_date, 'dd/MM/yyyy')")))

#Extraindo o ano da data convertida
df_info_silver = df_info_silver.withColumn(
    "ano_lancamento",
    F.year("data_lancamento"))

display(df_info_silver.select("release_date", "data_lancamento", "ano_lancamento").limit(10))

In [0]:
#Deixando somente as colunas necessárias, com os nomes e tipos definidos conforme o exercício
df_info_silver = df_info_silver.select(
    F.col("id").alias("id_filme"),
    F.col("title").alias("titulo"),
    F.col("original_title").alias("titulo_original"),
    F.col("data_lancamento"),
    F.expr("try_cast(runtime as int)").alias("duracao_minutos"),
    F.col("original_language").alias("idioma_original"),
    F.col("status_filme"),
    F.col("overview").alias("sinopse"),
    F.col("tagline").alias("frase_divulgacao"),
    F.col("ano_lancamento"))

#Salvando a tabela tratada na camada Silver
df_info_silver.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("workspace.silver.tb_info_filmes")

display(spark.table("workspace.silver.tb_info_filmes").limit(10))

In [0]:
#Lendo os dados financeiros brutos para iniciar o tratamento na Silver
df_financeiro = spark.table("workspace.bronze.tb_financeiro_filmes")

display(df_financeiro.limit(10))

In [0]:
#Removendo caracteres que não fazem parte do valor numérico antes da conversão. Texto de ausência e valores que não possam ser convertidos acabam como null
df_financeiro_silver = (
    df_financeiro
    .withColumn(
        "orcamento_temp",
        F.expr("try_cast(regexp_replace(budget, '[^0-9.-]', '') as decimal(18,2))"))
    .withColumn(
        "receita_temp",
        F.expr("try_cast(regexp_replace(revenue, '[^0-9.-]', '') as decimal(18,2))")))

#Como orçamento e receita precisam ser positivos para terem interpretação financeira, por isso zero e negativos ficam nulos
df_financeiro_silver = (
    df_financeiro_silver
    .withColumn(
        "orcamento_usd",
        F.when(F.col("orcamento_temp") > 0, F.col("orcamento_temp")))
    .withColumn(
        "receita_usd",
        F.when(F.col("receita_temp") > 0, F.col("receita_temp")))
    .select("id", "orcamento_usd", "receita_usd")
    .withColumnRenamed("id", "id_filme"))

display(df_financeiro_silver.limit(10))

In [0]:
#Usando a cotação de compra mais recente do período consultado
df_cotacao = (
    spark.table("workspace.bronze.tb_cotacao_dolar")
    .orderBy(F.col("dataHoraCotacao").desc())
    .select(F.col("cotacaoCompra").cast("decimal(10,4)").alias("cotacao_usd_brl"))
    .limit(1))

#Aplicando a cotação e calculo de lucro e margem. Se algum dos valores estiver ausente, o lucro também fica nulo
df_financeiro_silver = (
    df_financeiro_silver
    .crossJoin(df_cotacao)
    .withColumn("orcamento_brl", F.col("orcamento_usd") * F.col("cotacao_usd_brl"))
    .withColumn("receita_brl", F.col("receita_usd") * F.col("cotacao_usd_brl"))
    .withColumn("lucro_usd", F.col("receita_usd") - F.col("orcamento_usd"))
    .withColumn("lucro_brl", F.col("receita_brl") - F.col("orcamento_brl"))
    .withColumn(
        "margem_lucro_percentual",
        F.when(
            F.col("receita_usd").isNotNull() & (F.col("receita_usd") != 0),
            (F.col("lucro_usd") / F.col("receita_usd")) * 100
        )
    )
    .drop("cotacao_usd_brl"))

display(df_financeiro_silver.limit(10))

In [0]:
#Salvando os dados financeiros tratados na Silver
df_financeiro_silver.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("workspace.silver.tb_financeiro_filmes")

display(spark.table("workspace.silver.tb_financeiro_filmes").limit(10))

In [0]:
#Lendo as métricas brutas para verificar os valores antes da conversão
df_metricas = spark.table("workspace.bronze.tb_metricas_engajamento")

display(df_metricas.limit(10))

In [0]:
#Convertendo os campos com segurança, textos incompatíveis viram null. Utilizo try_cast para que textos deslocados virem nulos sem quebrar a execução
df_metricas_silver = (
    df_metricas
    .withColumn(
        "popularidade_temp",
        F.expr("try_cast(regexp_replace(popularity, ',', '.') as double)"))
    .withColumn("nota_tmdb_temp", F.expr("try_cast(vote_average as double)"))
    .withColumn("votos_tmdb_temp", F.expr("try_cast(vote_count as int)"))
    .withColumn("nota_imdb_temp", F.expr("try_cast(averageRating as double)"))
    .withColumn("votos_imdb_temp", F.expr("try_cast(numVotes as int)")))

#Considerei como deslocamento os valores com cara de ano quando as colunas de IMDb também têm texto incompatível
# Desconsidero anos que caíram na coluna de popularidade por deslocamento
df_metricas_silver = df_metricas_silver.withColumn(
    "popularidade",
    F.when(
        F.col("popularidade_temp").between(1900, 2100)
        & (
            (
                F.col("averageRating").isNotNull()
                & F.col("nota_imdb_temp").isNull()
            )
            | (
                F.col("numVotes").isNotNull()
                & F.col("votos_imdb_temp").isNull()
            )
        ),
        F.lit(None).cast("double")
    ).otherwise(
        F.when(F.col("popularidade_temp") >= 0, F.col("popularidade_temp"))))

#Notas só são consideradas válidas na escala de 0 a 10; contagens e popularidade negativas não fazem sentido e são anuladas

df_metricas_silver = df_metricas_silver.select(
    F.col("id").alias("id_filme"),
    "popularidade",
    F.when(F.col("nota_tmdb_temp").between(0, 10), F.col("nota_tmdb_temp")).alias("nota_media_tmdb"),
    F.when(F.col("votos_tmdb_temp") >= 0, F.col("votos_tmdb_temp")).alias("qtd_votos_tmdb"),
    F.when(F.col("nota_imdb_temp").between(0, 10), F.col("nota_imdb_temp")).alias("nota_media_imdb"),
    F.when(F.col("votos_imdb_temp") >= 0, F.col("votos_imdb_temp")).alias("qtd_votos_imdb"))

display(df_metricas_silver.limit(10))

In [0]:
display(
    df_metricas_silver
    .filter(F.col("id_filme").isin("809905", "658829", "557809", "418746")))

In [0]:
df_metricas_silver.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("workspace.silver.tb_metricas_engajamento")

In [0]:
#Lendo as avaliações brutas para iniciar o tratamento na Silver
df_avaliacoes = spark.table("workspace.bronze.tb_avaliacoes_usuarios")

display(df_avaliacoes.limit(10))

In [0]:
#Convertendo a nota e traduzindo os nomes das colunas
df_avaliacoes_silver = df_avaliacoes.select(
    F.col("id").alias("id_filme"),
    F.col("nome").alias("nome_usuario"),
    F.expr("try_cast(nota as double)").alias("nota_usuario"),
    F.when(
        F.col("comentario").isNull() | (F.trim(F.col("comentario")) == ""),
        "Sem comentário"
    ).otherwise(F.trim(F.col("comentario"))).alias("comentario"))

#Notas fora do intervalo de 0 a 10 não são consideradas válidas
df_avaliacoes_silver = df_avaliacoes_silver.withColumn(
    "nota_usuario",
    F.when(F.col("nota_usuario").between(0, 10), F.col("nota_usuario")))

#Considerei duplicada a avaliação do mesmo filme, usuário, nota e comentário
#Removendo avaliações repetidas
df_avaliacoes_silver = df_avaliacoes_silver.dropDuplicates(
    ["id_filme", "nome_usuario", "nota_usuario", "comentario"])

display(df_avaliacoes_silver.limit(10))

In [0]:
#Salvando as avaliações tratadas na camada Silver
df_avaliacoes_silver.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("workspace.silver.tb_avaliacoes_usuarios")

#Lendo os créditos e tags para preparar as próximas tabelas
df_creditos = spark.table("workspace.bronze.tb_creditos_tags")

display(df_creditos.select("genres").limit(10))

In [0]:
#Padronizando os separadores e separando os gêneros por linhas. Unifiquei os separadores antes do split para que cada gênero fique em uma linha
df_generos_silver = (
    df_creditos
    .select(
        "id",
        F.explode(
            F.split(F.regexp_replace(F.col("genres"), r"[;|]", ","), ",")
        ).alias("nome_genero")
    )
    .withColumn("nome_genero", F.initcap(F.trim("nome_genero"))))

#Mantendo os gêneros válidos e retirando repetições do mesmo filme
generos_validos = [
    "Action", "Adventure", "Animation", "Comedy", "Crime",
    "Documentary", "Drama", "Family", "Fantasy", "History",
    "Horror", "Music", "Mystery", "Romance", "Science Fiction",
    "Tv Movie", "Thriller", "War", "Western"]

df_generos_silver = (
    df_generos_silver
    .filter(F.col("nome_genero").isin(generos_validos))
    .withColumnRenamed("id", "id_filme")
    .dropDuplicates(["id_filme", "nome_genero"]))

display(df_generos_silver.limit(10))

In [0]:
#Salvando os gêneros tratados
df_generos_silver.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("workspace.silver.tb_generos")


#Separando cada tipo de entidade em linhas. Utilizo a mesma regra para atores, diretores, roteiristas e produtoras. Removo vazios e resíduos numéricos ou marcadores de ausência
def separar_entidades(coluna, tipo):
    return (
        df_creditos
        .select(
            F.col("id").alias("id_filme"),
            F.explode(
                F.split(F.regexp_replace(F.col(coluna), r"[;|]", ","), ",")
            ).alias("nome_entidade")
        )
        .withColumn("nome_entidade", F.initcap(F.trim("nome_entidade")))
        .withColumn("tipo_entidade", F.lit(tipo))
        .filter(
            F.col("nome_entidade").isNotNull()
            & (F.col("nome_entidade") != "")
            & (~F.col("nome_entidade").rlike("^[0-9]+$"))
            & (~F.upper(F.col("nome_entidade")).isin("N/A", "NULL", "UNKNOWN"))
        ))

atores = separar_entidades("cast", "Ator")
diretores = separar_entidades("directors", "Diretor")
roteiristas = separar_entidades("writers", "Roteirista")
produtoras = separar_entidades("production_companies", "Produtora")

df_pessoas_empresas_silver = (
    atores
    .unionByName(diretores)
    .unionByName(roteiristas)
    .unionByName(produtoras)
    .dropDuplicates(["id_filme", "nome_entidade", "tipo_entidade"]))

display(df_pessoas_empresas_silver.limit(10))

In [0]:
#Salvando as pessoas e empresas tratadas na Silver
df_pessoas_empresas_silver.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("workspace.silver.tb_pessoas_empresas")


#Lendo as cotações e convertendo a data e o valor
df_cotacoes = (
    spark.table("workspace.bronze.tb_cotacao_dolar")
    .withColumn("data_cotacao", F.to_date("dataHoraCotacao"))
    .withColumn("cotacao_compra", F.col("cotacaoCompra").cast("double")))

#Se houver mais de uma cotação no mesmo dia, mantenho a mais recente
janela_cotacao = Window.partitionBy("data_cotacao").orderBy(
    F.col("dataHoraCotacao").desc())

df_cotacoes = (
    df_cotacoes
    .withColumn("ordem", F.row_number().over(janela_cotacao))
    .filter(F.col("ordem") == 1)
    .select("data_cotacao", "cotacao_compra"))

#Completo finais de semana e feriados com a última cotação anterior, sem inventar uma nova cotação
#Criando uma linha para cada dia entre a primeira e a última cotação
limites = df_cotacoes.select(
    F.min("data_cotacao").alias("inicio"),
    F.max("data_cotacao").alias("fim")
).first()

df_calendario = spark.sql(
    f"SELECT explode(sequence(DATE('{limites['inicio']}'), "
    f"DATE('{limites['fim']}'), INTERVAL 1 DAY)) AS data_cotacao")

#Fazendi o forward fill para repetir a última cotação nos dias sem registro
df_cotacao_silver = (
    df_calendario
    .join(df_cotacoes, on="data_cotacao", how="left")
    .withColumn(
        "cotacao_compra",
        F.last("cotacao_compra", ignorenulls=True).over(
            Window.orderBy("data_cotacao")
            .rowsBetween(Window.unboundedPreceding, Window.currentRow))))

display(df_cotacao_silver)

In [0]:
#Salvando a série diária de cotações na camada Silver
df_cotacao_silver.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("workspace.silver.tb_cotacao_dolar")

display(spark.table("workspace.silver.tb_cotacao_dolar"))